In [15]:
import pandas as pd
import numpy as np
from pathlib import Path


In [ ]:
##i want to get coverage info for how many bases have 100X coverage across both reps
##going to try to get this info from the depth files

In [16]:
#transform txt to tsv and add headers
# root = Path("/Users/asjaeger/Desktop/project_source/hpai-PA/seq/illumina-pipeline/data")
# root = Path("/Users/asjaeger/Desktop/project_source/hpai-PA/seq/data")
root = Path("/Users/asjaeger/Desktop/project_source/hpai-PA/hpai-comm-pa/all_com_analysis_2pct/data")


header = "gene\tpos\tcvg\n"

for txt_file in root.rglob("remapping-2/depth.txt"):
    tsv_file = txt_file.with_suffix(".tsv")

    with txt_file.open("r") as fin, tsv_file.open("w") as fout:
        fout.write(header)
        for line in fin:
            fout.write("\t".join(line.split()) + "\n")

    print(f"Converted with header: {txt_file} → {tsv_file}")


Converted with header: /Users/asjaeger/Desktop/project_source/hpai-PA/hpai-comm-pa/all_com_analysis_2pct/data/run6_PCR_NC_4/replicate-1/remapping-2/depth.txt → /Users/asjaeger/Desktop/project_source/hpai-PA/hpai-comm-pa/all_com_analysis_2pct/data/run6_PCR_NC_4/replicate-1/remapping-2/depth.tsv
Converted with header: /Users/asjaeger/Desktop/project_source/hpai-PA/hpai-comm-pa/all_com_analysis_2pct/data/run4_RT_NC_1/replicate-1/remapping-2/depth.txt → /Users/asjaeger/Desktop/project_source/hpai-PA/hpai-comm-pa/all_com_analysis_2pct/data/run4_RT_NC_1/replicate-1/remapping-2/depth.tsv
Converted with header: /Users/asjaeger/Desktop/project_source/hpai-PA/hpai-comm-pa/all_com_analysis_2pct/data/run6_PCR_NC_3/replicate-1/remapping-2/depth.txt → /Users/asjaeger/Desktop/project_source/hpai-PA/hpai-comm-pa/all_com_analysis_2pct/data/run6_PCR_NC_3/replicate-1/remapping-2/depth.tsv
Converted with header: /Users/asjaeger/Desktop/project_source/hpai-PA/hpai-comm-pa/all_com_analysis_2pct/data/d_lbm2/

In [17]:
##merge rep 1 and rep 2 depth files for each sample
# root = Path("/Users/asjaeger/Desktop/project_source/hpai-PA/seq/data")

for sample_dir in root.iterdir():
    if not sample_dir.is_dir():
        continue

    dfs = []

    for cov_file in sample_dir.rglob("remapping-2/depth.tsv"):
        # Expect: sample/replicate-X/remap/coverage.tsv
        try:
            rep = cov_file.parents[1].name
        except IndexError:
            continue

        df = pd.read_csv(cov_file, sep="\t")
        df["rep"] = rep
        dfs.append(df)

    if not dfs:
        continue

    merged = pd.concat(dfs, ignore_index=True)

    out_file = sample_dir / "depth-reps.tsv"
    merged.to_csv(out_file, sep="\t", index=False)

    print(f"Wrote {out_file}")


Wrote /Users/asjaeger/Desktop/project_source/hpai-PA/hpai-comm-pa/all_com_analysis_2pct/data/run6_PCR_NC_4/depth-reps.tsv
Wrote /Users/asjaeger/Desktop/project_source/hpai-PA/hpai-comm-pa/all_com_analysis_2pct/data/run4_RT_NC_1/depth-reps.tsv
Wrote /Users/asjaeger/Desktop/project_source/hpai-PA/hpai-comm-pa/all_com_analysis_2pct/data/run6_PCR_NC_3/depth-reps.tsv
Wrote /Users/asjaeger/Desktop/project_source/hpai-PA/hpai-comm-pa/all_com_analysis_2pct/data/d_lbm2/depth-reps.tsv
Wrote /Users/asjaeger/Desktop/project_source/hpai-PA/hpai-comm-pa/all_com_analysis_2pct/data/cl_c2/depth-reps.tsv
Wrote /Users/asjaeger/Desktop/project_source/hpai-PA/hpai-comm-pa/all_com_analysis_2pct/data/run6_PCR_NC_2/depth-reps.tsv
Wrote /Users/asjaeger/Desktop/project_source/hpai-PA/hpai-comm-pa/all_com_analysis_2pct/data/run6_PCR_NC_5/depth-reps.tsv
Wrote /Users/asjaeger/Desktop/project_source/hpai-PA/hpai-comm-pa/all_com_analysis_2pct/data/d_lbm3/depth-reps.tsv
Wrote /Users/asjaeger/Desktop/project_source/hp

In [18]:
gene_lengths = pd.Series({
    "ha": 1776,
    "mp": 1027,
    "na": 1457,
    "np": 1565,
    "ns": 890,
    "pa": 2233,
    "pb1": 2341,
    "pb2": 2341,
})

In [19]:
##turning into function:

def coverage_pct_per_gene(sample_dir):
    df = pd.read_csv(
        Path(sample_dir) / "depth-reps.tsv",
        sep="\t"
    )

    df = (
        df
        .assign(rep=df["rep"].str.replace("replicate-", "rep"))
        .pivot(index=["gene", "pos"], columns="rep", values="cvg")
        .rename(columns=lambda c: f"cvg_{c}")
        .reset_index()
    )
    # ensure both replicates exist
    if not {"cvg_rep1", "cvg_rep2"}.issubset(df.columns):
        raise ValueError("Missing one or more replicates")

    # filter to >=100X in both replicates
    df = df[
        (df["cvg_rep1"] >= 100) &
        (df["cvg_rep2"] >= 100)
    ]

    observed = df["gene"].value_counts()

    pct = (observed / gene_lengths).fillna(0)

    return pct.to_dict()

In [20]:
##for looping through:

coverage_dict = {}

for sample_dir in root.iterdir():
    if not sample_dir.is_dir():
        continue

    try:
        coverage_dict[sample_dir.name] = coverage_pct_per_gene(sample_dir)
    except (FileNotFoundError, ValueError) as e:
        print(f"Skipping {sample_dir.name}: {e}")
##print statements mostly confirm that NCs and reference files aren't being used here in the loop

Skipping run6_PCR_NC_4: Missing one or more replicates
Skipping run4_RT_NC_1: Missing one or more replicates
Skipping run6_PCR_NC_3: Missing one or more replicates
Skipping cl_c2: Missing one or more replicates
Skipping run6_PCR_NC_2: Missing one or more replicates
Skipping run6_PCR_NC_5: Missing one or more replicates
Skipping rt_nc_run9: Missing one or more replicates
Skipping PCRNC4_run9: Missing one or more replicates
Skipping rt_nc3_run10: Missing one or more replicates
Skipping run5_PCR_NC_1: Missing one or more replicates
Skipping run4_PCR_NC_1: Missing one or more replicates
Skipping run5_RT_NC_1: Missing one or more replicates
Skipping PCR_NC1_run9: Missing one or more replicates
Skipping PCRNC3_run9: Missing one or more replicates
Skipping protein: [Errno 2] No such file or directory: '/Users/asjaeger/Desktop/project_source/hpai-PA/hpai-comm-pa/all_com_analysis_2pct/data/protein/depth-reps.tsv'
Skipping RTNC2_run8: Missing one or more replicates
Skipping PCRNC2_run8: Missing 

In [21]:
coverage_dict

{'d_lbm2': {'ha': 1.0,
  'mp': 0.0,
  'na': 0.8126286890871655,
  'np': 0.9731629392971246,
  'ns': 1.0,
  'pa': 0.5638154948499776,
  'pb1': 1.0,
  'pb2': 0.5296881674498077},
 'd_lbm3': {'ha': 1.0,
  'mp': 0.022395326192794548,
  'na': 1.0,
  'np': 0.9808306709265175,
  'ns': 1.0,
  'pa': 0.5642633228840125,
  'pb1': 1.0,
  'pb2': 1.0},
 'd_lbm4': {'ha': 1.0,
  'mp': 0.9707887049659202,
  'na': 0.9958819492107069,
  'np': 0.9993610223642172,
  'ns': 1.0,
  'pa': 0.9973130317957905,
  'pb1': 1.0,
  'pb2': 0.9953011533532679},
 'kc_com3': {'ha': 1.0,
  'mp': 0.9084712755598832,
  'na': 1.0,
  'np': 0.9993610223642172,
  'ns': 1.0,
  'pa': 1.0,
  'pb1': 1.0,
  'pb2': 1.0},
 'kc_com4': {'ha': 1.0,
  'mp': 0.9191820837390458,
  'na': 1.0,
  'np': 0.9936102236421726,
  'ns': 1.0,
  'pa': 0.999552171965965,
  'pb1': 1.0,
  'pb2': 0.99017513882956},
 'd_com3': {'ha': 1.0,
  'mp': 0.029211295034079845,
  'na': 0.9450926561427591,
  'np': 0.0,
  'ns': 1.0,
  'pa': 0.5409762651141962,
  'pb1': 

In [22]:
# Multiply values by gene lengths
sample_cvg_nt = {
    sample: {gene: value * gene_lengths[gene] for gene, value in genes.items()}
    for sample, genes in coverage_dict.items()
}

# Optionally, convert to integers if you want nucleotide counts
sample_cvg_nt_int = {
    sample: {gene: int(round(value)) for gene, value in genes.items()}
    for sample, genes in sample_cvg_nt.items()
}

print(sample_cvg_nt_int)

{'d_lbm2': {'ha': 1776, 'mp': 0, 'na': 1184, 'np': 1523, 'ns': 890, 'pa': 1259, 'pb1': 2341, 'pb2': 1240}, 'd_lbm3': {'ha': 1776, 'mp': 23, 'na': 1457, 'np': 1535, 'ns': 890, 'pa': 1260, 'pb1': 2341, 'pb2': 2341}, 'd_lbm4': {'ha': 1776, 'mp': 997, 'na': 1451, 'np': 1564, 'ns': 890, 'pa': 2227, 'pb1': 2341, 'pb2': 2330}, 'kc_com3': {'ha': 1776, 'mp': 933, 'na': 1457, 'np': 1564, 'ns': 890, 'pa': 2233, 'pb1': 2341, 'pb2': 2341}, 'kc_com4': {'ha': 1776, 'mp': 944, 'na': 1457, 'np': 1555, 'ns': 890, 'pa': 2232, 'pb1': 2341, 'pb2': 2318}, 'd_com3': {'ha': 1776, 'mp': 30, 'na': 1377, 'np': 0, 'ns': 890, 'pa': 1208, 'pb1': 1498, 'pb2': 2337}, 'kc_com5': {'ha': 1776, 'mp': 963, 'na': 1456, 'np': 1564, 'ns': 890, 'pa': 2233, 'pb1': 2341, 'pb2': 2321}, 'kc_com2': {'ha': 1776, 'mp': 972, 'na': 1457, 'np': 1564, 'ns': 890, 'pa': 2233, 'pb1': 1482, 'pb2': 2341}, 'd_com2': {'ha': 1776, 'mp': 0, 'na': 1377, 'np': 1535, 'ns': 889, 'pa': 2233, 'pb1': 2341, 'pb2': 2340}, 'cb_com1': {'ha': 1776, 'mp': 99

In [9]:
#export coverage dict
import json

with open("../all_com_analysis_2pct/data/sample_cvg_nt.json", "w") as f:
    json.dump(coverage_dict, f, indent=2)

In [3]:
##read in aa cvgs, from permutation script
import json

# with open("../../aa_cvg_test.json") as f:
with open("../all_com_analysis_2pct/data/allcom_cvg.json") as f:

    aa_cvg = json.load(f)
aa_cvg

{'c_com5': {'PB2': 760,
  'PB1': 758,
  'PA': 717,
  'HA': 568,
  'NP': 499,
  'NA': 470,
  'M1': 253,
  'M2': 327,
  'NS1': 231,
  'NEP': 279},
 'c_com6': {'PB2': 760,
  'PB1': 758,
  'PA': 717,
  'HA': 568,
  'NP': 499,
  'NA': 470,
  'M1': 253,
  'M2': 327,
  'NS1': 231,
  'NEP': 279},
 'c_lbm10': {'PB2': 760,
  'PB1': 758,
  'PA': 717,
  'HA': 568,
  'NP': 499,
  'NA': 470,
  'M1': 253,
  'M2': 327,
  'NS1': 231,
  'NEP': 279},
 'c_lbm11': {'PB2': 760,
  'PB1': 758,
  'PA': 717,
  'HA': 568,
  'NP': 499,
  'NA': 470,
  'M1': 253,
  'M2': 327,
  'NS1': 231,
  'NEP': 279},
 'c_lbm1': {'PB2': 760,
  'PB1': 758,
  'PA': 717,
  'HA': 568,
  'NP': 499,
  'NA': 470,
  'M1': 253,
  'M2': 327,
  'NS1': 231,
  'NEP': 279},
 'c_lbm2': {'PB2': 760,
  'PB1': 758,
  'PA': 717,
  'HA': 568,
  'NP': 499,
  'NA': 470,
  'M1': 253,
  'M2': 327,
  'NS1': 231,
  'NEP': 279},
 'c_lbm3': {'PB2': 760,
  'PB1': 758,
  'PA': 717,
  'HA': 568,
  'NP': 499,
  'NA': 470,
  'M1': 253,
  'M2': 327,
  'NS1': 231

In [4]:
##read in coverage_dict if already produced

# with open("../../aa_cvg_test.json") as f:
with open("../../seq/all_com_analysis/data/sample_cvg.json") as f:

    coverage_dict = json.load(f)
coverage_dict



{'d_lbm2': {'ha': 1.0,
  'mp': 0.0,
  'na': 0.8126286890871655,
  'np': 0.9731629392971246,
  'ns': 1.0,
  'pa': 0.5638154948499776,
  'pb1': 1.0,
  'pb2': 0.5296881674498077},
 'd_lbm3': {'ha': 1.0,
  'mp': 0.022395326192794548,
  'na': 1.0,
  'np': 0.9808306709265175,
  'ns': 1.0,
  'pa': 0.5642633228840125,
  'pb1': 1.0,
  'pb2': 1.0},
 'd_lbm4': {'ha': 1.0,
  'mp': 0.9707887049659202,
  'na': 0.9958819492107069,
  'np': 0.9993610223642172,
  'ns': 1.0,
  'pa': 0.9973130317957905,
  'pb1': 1.0,
  'pb2': 0.9953011533532679},
 'kc_com3': {'ha': 1.0,
  'mp': 0.9084712755598832,
  'na': 1.0,
  'np': 0.9993610223642172,
  'ns': 1.0,
  'pa': 1.0,
  'pb1': 1.0,
  'pb2': 1.0},
 'kc_com4': {'ha': 1.0,
  'mp': 0.9191820837390458,
  'na': 1.0,
  'np': 0.9936102236421726,
  'ns': 1.0,
  'pa': 0.999552171965965,
  'pb1': 1.0,
  'pb2': 0.99017513882956},
 'd_com3': {'ha': 1.0,
  'mp': 0.029211295034079845,
  'na': 0.9450926561427591,
  'np': 0.0,
  'ns': 1.0,
  'pa': 0.5409762651141962,
  'pb1': 

In [23]:
gene_map = {
    "HA": "ha",
    "NA": "na",
    "NP": "np",
    "PA": "pa",
    "PB1": "pb1",
    "PB2": "pb2",
    "NS1": "ns",
    "NEP": "ns",
    "M1": "mp",
    "M2": "mp",
}

# count_dict → what’s there, subset here!

# coverage_dict → the fraction that has coverage dict made here

# gene_map → how names correspond

# sample_cvg → QC-adjusted result

sample_cvg = {}

for sample, counts in aa_cvg.items():
    if sample not in coverage_dict:
        continue

    sample_cvg[sample] = {}

    for gene, count in counts.items():
        gene_upper = gene.upper()

        if gene_upper not in gene_map:
            continue

        cov_key = gene_map[gene_upper]

        cov_frac = coverage_dict[sample].get(cov_key, 0)

        sample_cvg[sample][gene_upper] = cov_frac * count
sample_cvg

{'c_com5': {'PB2': 759.6753524134986,
  'PB1': 758.0,
  'PA': 717.0,
  'HA': 568.0,
  'NP': 497.405750798722,
  'NA': 470.0,
  'M1': 238.2190847127556,
  'M2': 307.8958130477118,
  'NS1': 230.74044943820223,
  'NEP': 278.6865168539326},
 'c_com6': {'PB2': 759.6753524134986,
  'PB1': 758.0,
  'PA': 717.0,
  'HA': 568.0,
  'NP': 498.6811501597444,
  'NA': 470.0,
  'M1': 234.03115871470303,
  'M2': 302.4829600778968,
  'NS1': 231.0,
  'NEP': 279.0},
 'c_lbm10': {'PB2': 760.0,
  'PB1': 758.0,
  'PA': 717.0,
  'HA': 568.0,
  'NP': 495.49265175718847,
  'NA': 470.0,
  'M1': 240.43622200584227,
  'M2': 310.761441090555,
  'NS1': 231.0,
  'NEP': 279.0},
 'c_lbm11': {'PB2': 0.0,
  'PB1': 735.0106791969243,
  'PA': 404.25570980743396,
  'HA': 566.4009009009009,
  'NP': 0.0,
  'NA': 0.0,
  'M1': 0.0,
  'M2': 0.0,
  'NS1': 229.70224719101122,
  'NEP': 277.4325842696629},
 'c_lbm1': {'PB2': 441.1960700555319,
  'PB1': 758.0,
  'PA': 715.0734437975817,
  'HA': 568.0,
  'NP': 488.7968051118211,
  'NA

In [14]:
##export aa cvg updated
with open("../all_com_analysis_2pct/data/sample_cvg_aa_merged.json", "w") as f:
    json.dump(sample_cvg, f, indent=2)

In [ ]:
##so basically this should work to replace the aa_coverage dicts in the permutations script
##so once i have depth.txts outputting again, can try outputting 

In [25]:
df = pd.read_csv('../output_dfs/2026-05-05_repshared_SNVs_allcom_lbm_2pct.tsv', sep='\t')
df['gene'] = df['gene'].fillna('NA')

print(df['gene'].unique())

['NA' 'PA' 'PB1' 'NP' 'PB2' 'HA' 'NS1' 'M2' 'M1' 'NEP']


In [26]:
df

,segment,position,allele,coding_region_change,gene,Frequency_1,Frequency_2,sample,amino_acid,gene_pos,...,ct_value,rep_shared,order,county,date,species,species_group,color,avg_freq,order_domstat
0,na,43,T,Ile8Thr,NA,0.3326,0.3412,c_com5,8,NA_8,...,22.01,shared,galliformes,unknown,2025-01-26,NaN,Galliformes,commercial_nonsynonymous,0.33690,galliformes_commercial
1,na,67,T,Val16Ala,NA,0.2439,0.2890,c_com5,16,NA_16,...,22.01,shared,galliformes,unknown,2025-01-26,NaN,Galliformes,commercial_nonsynonymous,0.26645,galliformes_commercial
2,na,76,T,Ile19Thr,NA,0.2454,0.2615,c_com5,19,NA_19,...,22.01,shared,galliformes,unknown,2025-01-26,NaN,Galliformes,commercial_nonsynonymous,0.25345,galliformes_commercial
3,na,79,T,Ile20Thr,NA,0.2392,0.2624,c_com5,20,NA_20,...,22.01,shared,galliformes,unknown,2025-01-26,NaN,Galliformes,commercial_nonsynonymous,0.25080,galliformes_commercial
4,pa,1173,C,Asp383Asp,PA,0.1338,0.1468,c_com5,383,PA_383,...,22.01,shared,galliformes,unknown,2025-01-26,NaN,Galliformes,commercial_synonymous,0.14030,galliformes_commercial
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
302,pb2,1560,A,Val511Val,PB2,0.0869,0.0918,kc_com6,511,PB2_511,...,21.93,shared,anseriformes,Chester,2023-02-23,duck,Anseriformes,commercial_synonymous,0.08935,anseriformes_commercial
303,pb2,1659,A,Ser544Ser,PB2,0.4205,0.4155,kc_com6,544,PB2_544,...,21.93,shared,anseriformes,Chester,2023-02-23,duck,Anseriformes,commercial_synonymous,0.41800,anseriformes_commercial
304,np,1023,T,Ser326Ser,NP,0.1652,0.0818,kc_com7,326,NP_326,...,22.13,shared,anseriformes,Chester,2023-02-23,duck,Anseriformes,commercial_synonymous,0.12350,anseriformes_commercial
305,pb2,1659,A,Ser544Ser,PB2,0.3472,0.3537,kc_com7,544,PB2_544,...,22.13,shared,anseriformes,Chester,2023-02-23,duck,Anseriformes,commercial_synonymous,0.35045,anseriformes_commercial


In [27]:
df['segment'].unique()

array(['na', 'pa', 'pb1', 'np', 'pb2', 'ha', 'ns', 'mp'], dtype=object)

In [28]:
import re

SNPs = {}

for index, row in df.iterrows():
    sample = row['sample']
    aa_change = row['coding_region_change']
    syn_status = row['syn_non']
    gene = row['segment']

    # Extract amino acid site number
    match = re.search(r'(\d+)', str(aa_change))
    if match:
        aa_site = match.group(1)
    else:
        continue

    # Only count synonymous or nonsynonymous
    if syn_status not in ["synonymous", "nonsynonymous"]:
        continue

    # Initialize sample if needed
    if sample not in SNPs:
        SNPs[sample] = {g: [] for g in ["pb2", "pb1", "pa", "ha", "np", "na", "mp", "ns"]}

    if gene not in SNPs[sample]:
        continue

    # Add unique amino acid site
    if aa_site not in SNPs[sample][gene]:
        SNPs[sample][gene].append(aa_site)

# Convert to counts
SNP_counts = {
    sample: {gene: len(aa_sites) for gene, aa_sites in genes.items()}
    for sample, genes in SNPs.items()
}

print(SNP_counts)

{'c_com5': {'pb2': 0, 'pb1': 3, 'pa': 2, 'ha': 0, 'np': 0, 'na': 4, 'mp': 0, 'ns': 0}, 'c_com6': {'pb2': 1, 'pb1': 0, 'pa': 0, 'ha': 0, 'np': 1, 'na': 1, 'mp': 0, 'ns': 0}, 'c_lbm10': {'pb2': 1, 'pb1': 1, 'pa': 1, 'ha': 1, 'np': 2, 'na': 0, 'mp': 0, 'ns': 1}, 'c_lbm11': {'pb2': 0, 'pb1': 4, 'pa': 0, 'ha': 2, 'np': 0, 'na': 0, 'mp': 0, 'ns': 0}, 'c_lbm1': {'pb2': 0, 'pb1': 0, 'pa': 0, 'ha': 1, 'np': 2, 'na': 2, 'mp': 0, 'ns': 0}, 'c_lbm2': {'pb2': 3, 'pb1': 1, 'pa': 0, 'ha': 0, 'np': 3, 'na': 1, 'mp': 0, 'ns': 0}, 'c_lbm3': {'pb2': 0, 'pb1': 0, 'pa': 0, 'ha': 1, 'np': 0, 'na': 0, 'mp': 0, 'ns': 0}, 'c_lbm5': {'pb2': 2, 'pb1': 4, 'pa': 1, 'ha': 2, 'np': 2, 'na': 4, 'mp': 0, 'ns': 1}, 'c_lbm6': {'pb2': 0, 'pb1': 0, 'pa': 2, 'ha': 1, 'np': 2, 'na': 0, 'mp': 0, 'ns': 2}, 'c_lbm7': {'pb2': 1, 'pb1': 2, 'pa': 4, 'ha': 1, 'np': 0, 'na': 3, 'mp': 0, 'ns': 0}, 'c_lbm8': {'pb2': 1, 'pb1': 1, 'pa': 1, 'ha': 1, 'np': 4, 'na': 2, 'mp': 0, 'ns': 0}, 'c_lbm9': {'pb2': 2, 'pb1': 1, 'pa': 1, 'ha': 0, 'n

In [29]:
# Divide each value by the corresponding value in sample_cvg_nt_int
divided_values = {
    sample: {
        gene: SNP_counts[sample][gene] / sample_cvg_nt_int[sample][gene]
        if sample_cvg_nt_int[sample][gene] != 0 else 0
        for gene in SNP_counts[sample]
    }
    for sample in SNP_counts
}

# Optional: round to 4 decimals
divided_values_rounded = {
    sample: {gene: round(value, 4) for gene, value in genes.items()}
    for sample, genes in divided_values.items()
}

print(divided_values_rounded)

{'c_com5': {'pb2': 0.0, 'pb1': 0.0013, 'pa': 0.0009, 'ha': 0.0, 'np': 0.0, 'na': 0.0027, 'mp': 0.0, 'ns': 0.0}, 'c_com6': {'pb2': 0.0004, 'pb1': 0.0, 'pa': 0.0, 'ha': 0.0, 'np': 0.0006, 'na': 0.0007, 'mp': 0.0, 'ns': 0.0}, 'c_lbm10': {'pb2': 0.0004, 'pb1': 0.0004, 'pa': 0.0004, 'ha': 0.0006, 'np': 0.0013, 'na': 0.0, 'mp': 0.0, 'ns': 0.0011}, 'c_lbm11': {'pb2': 0, 'pb1': 0.0018, 'pa': 0.0, 'ha': 0.0011, 'np': 0, 'na': 0, 'mp': 0, 'ns': 0.0}, 'c_lbm1': {'pb2': 0.0, 'pb1': 0.0, 'pa': 0.0, 'ha': 0.0006, 'np': 0.0013, 'na': 0.0014, 'mp': 0, 'ns': 0.0}, 'c_lbm2': {'pb2': 0.0013, 'pb1': 0.0004, 'pa': 0.0, 'ha': 0.0, 'np': 0.002, 'na': 0.0007, 'mp': 0.0, 'ns': 0.0}, 'c_lbm3': {'pb2': 0.0, 'pb1': 0.0, 'pa': 0.0, 'ha': 0.0006, 'np': 0.0, 'na': 0, 'mp': 0, 'ns': 0.0}, 'c_lbm5': {'pb2': 0.0009, 'pb1': 0.0017, 'pa': 0.0004, 'ha': 0.0011, 'np': 0.0013, 'na': 0.0027, 'mp': 0.0, 'ns': 0.0011}, 'c_lbm6': {'pb2': 0.0, 'pb1': 0.0, 'pa': 0.0009, 'ha': 0.0006, 'np': 0.0013, 'na': 0.0, 'mp': 0, 'ns': 0.0022

In [31]:
# segment_to_gene = {
#     'ha': ['HA'],
#     'na': ['NA'],
#     'np': ['NP'],
#     'pa': ['PA'],
#     'pb1': ['PB1'],
#     'pb2': ['PB2'],
#     'mp': ['M1', 'M2'],
#     'ns': ['NS1', 'NEP']
# }
# snp_density = {}

# # Only use samples that exist in BOTH dictionaries
# common_samples = set(coverage_dict.keys()) & set(SNP_counts.keys())

# for sample in common_samples:
#     snp_density[sample] = {}

#     for segment, coverage_prop in coverage_dict[sample].items():

#         gene_len = gene_lengths.get(segment, 0)

#         # SNP dictionary uses uppercase gene names
#         snps = SNP_counts[sample].get(segment.upper(), 0)

#         if gene_len == 0 or coverage_prop == 0:
#             snp_density[sample][segment] = 0
#             continue

#         covered_nt = gene_len * coverage_prop
#         covered_kb = covered_nt / 1000

#         snp_density[sample][segment] = snps / covered_kb if covered_kb > 0 else 0

# for sample in snp_density:
#     for gene in snp_density[sample]:
#         snp_density[sample][gene] = round(float(snp_density[sample][gene]), 3)

# print(snp_density)

{'g_com1': {'ha': 0.0, 'mp': 0.0, 'na': 0.0, 'np': 0.0, 'ns': 0.0, 'pa': 0.885, 'pb1': 0.0, 'pb2': 0.0}, 'c_lbm8': {'ha': 0.563, 'mp': 0.0, 'na': 1.373, 'np': 2.606, 'ns': 0.0, 'pa': 0.449, 'pb1': 0.427, 'pb2': 0.427}, 'c_lbm11': {'ha': 1.129, 'mp': 0.0, 'na': 0.0, 'np': 0.0, 'ns': 0.0, 'pa': 0.0, 'pb1': 1.762, 'pb2': 0.0}, 'g_com2': {'ha': 0.0, 'mp': 0.0, 'na': 0.0, 'np': 0.0, 'ns': 0.0, 'pa': 0.474, 'pb1': 0.0, 'pb2': 1.605}, 'c_lbm10': {'ha': 1.126, 'mp': 0.0, 'na': 0.0, 'np': 1.287, 'ns': 0.0, 'pa': 0.896, 'pb1': 0.427, 'pb2': 0.427}, 'c_lbm3': {'ha': 0.563, 'mp': 0.0, 'na': 0.0, 'np': 0.0, 'ns': 0.0, 'pa': 0.0, 'pb1': 0.0, 'pb2': 0.0}, 'c_lbm5': {'ha': 1.126, 'mp': 0.0, 'na': 3.432, 'np': 1.288, 'ns': 0.0, 'pa': 0.448, 'pb1': 2.136, 'pb2': 2.563}, 'ckr_com2': {'ha': 0.563, 'mp': 0.0, 'na': 0.726, 'np': 2.564, 'ns': 0.0, 'pa': 0.448, 'pb1': 0.854, 'pb2': 1.709}, 'c_lbm2': {'ha': 0.564, 'mp': 0.0, 'na': 0.73, 'np': 2.009, 'ns': 0.0, 'pa': 0.0, 'pb1': 0.427, 'pb2': 1.735}, 'c_lbm7': 

In [ ]:
###OK NO THIS IS WRONG- what i want to do:
#take my gene lengths dict (nt length), multiply those lengths by the percentages of coverage for each gene for each sample,
##and then take those values (total nts with coverage) a

In [ ]:
# snp_densitySNP_counts

In [31]:
df_snp_cts = pd.DataFrame.from_dict(SNP_counts, orient='index')
df_snp_cts = df_snp_cts.reset_index()
df_snp_cts = df_snp_cts.rename(columns={'index': 'sample_id'})
df_snp_cts['total_snps'] = df_snp_cts.iloc[:, 1:9].sum(axis=1)

df_snp_cts

,sample_id,pb2,pb1,pa,ha,np,na,mp,ns,total_snps
0,c_com5,0,3,2,0,0,4,0,0,9
1,c_com6,1,0,0,0,1,1,0,0,3
2,c_lbm10,1,1,1,1,2,0,0,1,7
3,c_lbm11,0,4,0,2,0,0,0,0,6
4,c_lbm1,0,0,0,1,2,2,0,0,5
5,c_lbm2,3,1,0,0,3,1,0,0,8
6,c_lbm3,0,0,0,1,0,0,0,0,1
7,c_lbm5,2,4,1,2,2,4,0,1,16
8,c_lbm6,0,0,2,1,2,0,0,2,7
9,c_lbm7,1,2,4,1,0,3,0,0,11


In [32]:
df_snp_cvg_nt = pd.DataFrame.from_dict(sample_cvg_nt_int, orient='index')
df_snp_cvg_nt = df_snp_cvg_nt.reset_index()
df_snp_cvg_nt = df_snp_cvg_nt.rename(columns={'index': 'sample_id'})
df_snp_cvg_nt['total_nts'] = df_snp_cvg_nt.iloc[:, 1:9].sum(axis=1)

df_snp_cvg_nt

,sample_id,ha,mp,na,np,ns,pa,pb1,pb2,total_nts
0,d_lbm2,1776,0,1184,1523,890,1259,2341,1240,10213
1,d_lbm3,1776,23,1457,1535,890,1260,2341,2341,11623
2,d_lbm4,1776,997,1451,1564,890,2227,2341,2330,13576
3,kc_com3,1776,933,1457,1564,890,2233,2341,2341,13535
4,kc_com4,1776,944,1457,1555,890,2232,2341,2318,13513
5,d_com3,1776,30,1377,0,890,1208,1498,2337,9116
6,kc_com5,1776,963,1456,1564,890,2233,2341,2321,13544
7,kc_com2,1776,972,1457,1564,890,2233,1482,2341,12715
8,d_com2,1776,0,1377,1535,889,2233,2341,2340,12491
9,cb_com1,1776,995,1442,1564,890,2233,2341,2341,13582


In [33]:
nts_snps = pd.merge(df_snp_cvg_nt[['sample_id', 'total_nts']], df_snp_cts[['sample_id', 'total_snps']], on='sample_id', how='left')
nts_snps['snps_per_1kb'] = nts_snps['total_snps']/nts_snps['total_nts']*1000
nts_snps.fillna(0, inplace=True)
nts_snps

,sample_id,total_nts,total_snps,snps_per_1kb
0,d_lbm2,10213,13.0,1.272887
1,d_lbm3,11623,12.0,1.032436
2,d_lbm4,13576,10.0,0.736594
3,kc_com3,13535,13.0,0.960473
4,kc_com4,13513,5.0,0.370014
5,d_com3,9116,3.0,0.329092
6,kc_com5,13544,8.0,0.590667
7,kc_com2,12715,0.0,0.000000
8,d_com2,12491,12.0,0.960692
9,cb_com1,13582,11.0,0.809895


In [34]:
# nts_snps.to_csv('../../seq/analysis/output_dfs/snps-per-1kb-allcomm_2pct.tsv', sep='\t')
nts_snps.to_csv('../output_dfs/2026-05-05-snps-per-1kb-allcomm.tsv', sep='\t')